In [ ]:
# days_since_prior_order의 결측치를 -1 또는 신규고객(1/0) 변수로 변환.
# 데이터프레임 전체의 메모리 점유율을 확인하고 데이터 타입 최적화(Downcasting).
import pandas as pd
import numpy as np

# 데이터 로드
print("데이터를 불러오는 중입니다... 잠시만 기다려주세요.")

# 데이터 타입 최적화
def reduce_memory_usage(df, file_name=''):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        
        # 1. 정수형(int)인 경우
        if str(col_type)[:3] == 'int':
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
        
        # 2. 실수형(float)인 경우만 float32로 변경
        elif str(col_type)[:5] == 'float':
            df[col] = df[col].astype(np.float32)
            
        # 3. 그 외(object, string 등)는 무시하고 넘어감
        else:
            continue
            
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'메모리 최적화 완료: {start_mem:.2f}MB -> {end_mem:.2f}MB ({100*(start_mem-end_mem)/start_mem:.1f}% 감소)')
    return df

# 데이터 불러오기
orders = pd.read_csv('../data/orders.csv')  # 주문 정보
# 신규 고객 여부 변수 생성 (첫 주문: 1, 기존 고객의 재주문: 0)
orders['is_first_order'] = (orders['days_since_prior_order'].isna()).astype(np.int8)
orders['days_since_prior_order'] = orders['days_since_prior_order'].fillna(-1)  # 결측치 처리 (첫 주문(NaN)을 -1로 변경)
orders = reduce_memory_usage(orders, "orders") # 데이터 최적화
missing_count = orders['days_since_prior_order'].isnull().sum()   # 결측치가 0개인지 확인
print(f"\n결측치 개수: {missing_count}")

# 전처리된 데이터 저장
orders.to_csv('../data/orders_mem.csv', index=False)
del orders # 메모리 해제

# prior 데이터 프레임 메모리 최적화 및 저장
prior = pd.read_csv('../data/order_products__prior.csv')  # 상세 주문 내역
prior = reduce_memory_usage(prior, "prior")
prior.to_csv('../data/prior_mem.csv', index=False)
del prior # 메모리 해제

# 나머지 파일
file_list = {
    'order_products__train.csv': 'train_mem.csv',
    'products.csv': 'products_mem.csv',
    'aisles.csv': 'aisles_mem.csv',
    'departments.csv': 'departments_mem.csv'
}

for file_name, mem_file_name in file_list.items():
    print(f"\n{file_name} 파일을 처리 중입니다...")
    df = pd.read_csv(f'../data/{file_name}')
    df = reduce_memory_usage(df, file_name.split('.')[0])
    df.to_csv(f'../data/{mem_file_name}', index=False)
    del df # 메모리 해제
    
print("\n모든 데이터 파일의 메모리 최적화가 완료되었습니다.")

데이터를 불러오는 중입니다... 잠시만 기다려주세요.
메모리 최적화 완료: 185.97MB -> 78.30MB (57.9% 감소)

결측치 개수: 0
